# 第5回：何を、いつ、何のために予測するか

**今日の問い：モデル構築より前に決めるべきことは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 予測問題を1文にする

例：**実験条件を決める時点で利用できる情報から、収率を予測し、優先して実施する条件を選ぶ。**

`post_assay_signal`、`purity_pct`、`yield_pct`は実験後に得られるため、この時点の説明変数にはできません。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


## TRY：単純な予測を基準にする

複雑なモデルより先に、平均値または最頻値だけを返すモデルを作ります。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけで予測したMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけで予測した正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


## TRY：自分のテーマを整理する

次の7項目を埋めます。

1. 誰が、何の判断に使うか
2. いつ予測するか
3. 目的変数
4. その時点で利用できる説明変数
5. 利用してはいけない情報
6. 回帰か分類か
7. 単純な基準は何か

## ASK COPILOT

曖昧な点を推測で埋めず、確認質問として返すよう依頼します。
